In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    to_date,
    to_timestamp,
    when,
    current_timestamp,
    from_json
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    IntegerType
)


# ============================================
# STORAGE PATHS
# ============================================

bronze_root = (
    "abfss://bronze@bankingdelakevishal.dfs.core.windows.net/"
)

silver_root = (
    "abfss://silver@bankingdelakevishal.dfs.core.windows.net/"
)

In [0]:
# ============================================
# LOAN: BRONZE → SILVER
# ============================================
# CONFIGURATION & SCHEMA SETUP
# ============================================
CATALOG_NAME = "banking_lakehouse_db2"
SCHEMA_NAME = "silver"
TABLE_NAME = "loan"
FULL_TABLE_NAME = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{TABLE_NAME}"

# Ensure Silver schema exists inside your Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

# ============================================
# READ BRONZE DATA
# ============================================
loan_bronze_df = spark.read.parquet(f"{bronze_root}loan/")

# ============================================
# TRANSFORM & CLEANSE (BRONZE → SILVER)
# ============================================
loan_silver_df = (
    loan_bronze_df
    .select(
        trim(col("loan_id")).alias("loan_id"),
        trim(col("customer_id")).alias("customer_id"),
        upper(trim(col("loan_type"))).alias("loan_type"),
        trim(col("principal")).cast(DecimalType(18, 2)).alias("principal"),
        trim(col("interest_rate")).cast(DecimalType(5, 2)).alias("interest_rate"),
        trim(col("tenure_months")).cast(IntegerType()).alias("tenure_months"),
        upper(trim(col("loan_status"))).alias("loan_status")
    )
    .filter(col("loan_id").isNotNull())
    .dropDuplicates(["loan_id"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

# ============================================
# WRITE FRESH DATA (OVERWRITE PATH & TABLE)
# ============================================
# 1. Overwrite raw Delta files in ADLS
(
    loan_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{silver_root}loan/")
)

# 2. Overwrite / Register managed Unity Catalog table
(
    loan_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

# ============================================
# VERIFICATION METRICS
# ============================================
record_count = loan_silver_df.count()
print(f"Loan Silver Count: {record_count} records saved to '{FULL_TABLE_NAME}'.")